In [3]:
!pip install wandb onnx -Uq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 77.1 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Imports**

In [9]:
!pip install neuralforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.0/294.0 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 11.3 MB/s eta 0:00:00
  Attempting uninstall: pytorch-lightning
    Found existing installation: pytorch-lightning 2.6.5
    Uninstalling pytorch-lightning-2.6.5:
      Successfully uninstalled pytorch-lightning-2.6.5


In [10]:
import pandas as pd
import numpy as np
import torch


import numpy as np
import torch
import wandb


import torch
import pandas as pd
from neuralforecast.models import NBEATS
from neuralforecast import NeuralForecast
from sklearn.base import BaseEstimator, TransformerMixin

import wandb
import json


**WandB**

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: ERROR Invalid API key: API key must have 40+ characters, has 1.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tdola23 (tdola23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

**Reading Data**

In [16]:
train_df = pd.read_csv("/content/drive/MyDrive/WalmartPrices/NBeats/train_parquet.csv")
val_df = pd.read_csv("/content/drive/MyDrive/WalmartPrices/NBeats/val_parquet.csv")
X_train = train_df[["unique_id", "Date", "y", "IsHoliday"]].copy()
y_train = train_df["y"].copy()
X_val = val_df[["unique_id", "Date", "y", "IsHoliday"]].copy()
y_val = val_df["y"].copy()
X_train.head()

,unique_id,Date,y,IsHoliday
0,10_1,2010-02-05,40212.84,False
1,10_1,2010-02-12,67699.32,True
2,10_1,2010-02-19,49748.33,False
3,10_1,2010-02-26,33601.22,False
4,10_1,2010-03-05,36572.44,False


**WMAE LOSS**

In [17]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5.0, 1.0)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

**Dataset Prepare**

In [31]:
def prepare_df(df):
    df = df.copy()

    df["ds"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["unique_id", "ds"])

    return df.drop(columns=["Date"])

**Model**

In [33]:
def build_model(h, input_size, lr):

    model = NBEATS(
        input_size=input_size,
        h=h,
        max_steps=1500,
        batch_size=64,
        learning_rate=lr,
        stack_types=['identity', 'trend', 'seasonality'],
        n_blocks=[1, 1, 1],
        random_seed=42,
        start_padding_enabled=True
    )

    return NeuralForecast(
        models=[model],
        freq="W-FRI"
    )

In [29]:
def train_and_eval(train_df, val_df, config):

    wandb.init(project="walmart-sales-forecasting", config=config)

    train_df = prepare_df(X_train)
    print(train_df.head())
    val_df = prepare_df(X_val)
    print(val_df.head())
    nf = build_model(
        h=config["h"],
        input_size=config["input_size"],
        lr=config["lr"]
    )

    nf.fit(train_df)

    preds = nf.predict()

    # align predictions with validation
    merged = val_df.merge(
        preds,
        on=["unique_id", "ds"],
        how="left"
    )

    merged = merged.fillna(0)

    score = wmae(
        merged["y"].values,
        merged["NBEATS"].values,
        merged["IsHoliday"].values
    )

    wandb.log({"wmae": score})

    wandb.finish()

    return score

**Training Methods**

In [34]:
configs = [
    {"lr": lr, "h": h, "input_size": inp}
    for lr in [0.001, 0.0005, 0.0001]
    for h in [35]
    for inp in [20, 26, 36, 52]
]

best_score = float("inf")
best_config = None

for config in configs:

    score = train_and_eval(train_df, val_df, config)

    print(config, score)

    if score < best_score:
        best_score = score
        best_config = config

print("BEST:", best_config, best_score)

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True


  unique_id         y  IsHoliday         ds
0      10_1  40212.84      False 2010-02-05
1      10_1  67699.32       True 2010-02-12
2      10_1  49748.33      False 2010-02-19
3      10_1  33601.22      False 2010-02-26
4      10_1  36572.44      False 2010-03-05
  unique_id         y  IsHoliday         ds
0      10_1  30525.88      False 2012-03-02
1      10_1  33728.46      False 2012-03-09
2      10_1  34745.10      False 2012-03-16
3      10_1  38656.88      False 2012-03-23
4      10_1  46922.97      False 2012-03-30


INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.5 M  | train
-------------------------------------------------------
2.5 M     Trainable params
3.9 K     Non-trainable params
2.5 M     Total params
10.005    Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=1500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

wmae,▁
wmae,1663.64293


{'lr': 0.0001, 'h': 35, 'input_size': 52} 1663.642934295477
BEST: {'lr': 0.0005, 'h': 35, 'input_size': 52} 1612.8352767037284
